In [ ]:
# Tabela GOLD SDC 2
# Aplicação do SDC 2:
# Assim a tabela deixa de ter 1 linha por produto e passa a ter 1 linha por versão do produto.

# Antes ( SCD Tipo 1) = (1 produto = 1 registro)
# Depois (SCD Tipo 2) = (1 produto = N registros (versões históricas))
# Adicionadas novas colunas de controle de vigência
# É gerado uma nova 'sk_produto' para cada mudança
# Isso permite responder: “Como o produto era naquela data?”

## No SCD Tipo 2, a dimensão produto passa a armazenar múltiplas versões do mesmo produto,
# controladas por datas de vigência e uma chave substituta distinta, garantindo preservação histórica.”

# A dim_produto deixa de sobrescrever dados e passa a versionar cada mudança.




In [1]:
import pandas as pd
import sqlite3


In [2]:
# Um banco por projeto:

db_path = 'db_project_eng_dados'
conexao  = sqlite3.connect(database=db_path)

In [15]:
# Tabela de Classificação de Dados : Data Dictionary / Metadata 

conexao.execute("""
CREATE TABLE IF NOT EXISTS dict_class (
    nome_tabela TEXT,
    campo TEXT,
    papel TEXT,
    camada TEXT,
    observacao TEXT) """)

conexao.execute("""INSERT INTO dict_class(
    nome_tabela,
    campo,
    papel,
    camada,
    observacao) VALUES 
    ('silver_produtos','id_produto','Chave Natural','Silver','Identificador do produto na origem'),
    ('silver_produtos','nome_produto','Dimensão','Gold', 'Atributo descritivo do produto'),
    ('silver_produtos','tipo_dado','Dimensão','Gold', 'Tipo do dado geoespacial'),
    ('silver_produtos','sistema_coordenadas','Dimensão','Gold', 'Sistema de referência espacial'),
    ('silver_produtos','fornecedor','Dimensão','Gold', 'Fornecedor do dado'),
    ('silver_produtos','resolucao_espacial','Medida','Gold', 'Resolução espacial do produto'),
    ('silver_produtos','area_cobertura_km2','Medida','Gold', 'Área de cobertura em km²'),
    ('silver_produtos','preco','Medida','Gold', 'Valor comercial do produto'),        
    ('silver_produtos','data_bronze','Linhagem','Controle', 'Data de carga na camada Bronze'),  
    ('silver_produtos','data_silver','Linhagem','Controle', 'Data de carga na camada Silver');                                                                                                                                          
                
""")

conexao.commit()


In [4]:
# Mostragem da tabela de classificação de dados em SQLite:

cursor = conexao.cursor()

cursor.execute("SELECT * FROM dict_class;")
rows = cursor.fetchall()

for row in rows:
    print(row)


('silver_produtos', 'id_produto', 'Chave Natural', 'Silver', 'Identificador do produto na origem')
('silver_produtos', 'nome_produto', 'Dimensão', 'Gold', 'Atributo descritivo do produto')
('silver_produtos', 'tipo_dado', 'Dimensão', 'Gold', 'Tipo do dado geoespacial')
('silver_produtos', 'sistema_coordenadas', 'Dimensão', 'Gold', 'Sistema de referência espacial')
('silver_produtos', 'fornecedor', 'Dimensão', 'Gold', 'Fornecedor do dado')
('silver_produtos', 'resolucao_espacial', 'Medida', 'Gold', 'Resolução espacial do produto')
('silver_produtos', 'area_cobertura_km2', 'Medida', 'Gold', 'Área de cobertura em km²')
('silver_produtos', 'preco', 'Medida', 'Gold', 'Valor comercial do produto')
('silver_produtos', 'data_bronze', 'Linhagem', 'Controle', 'Data de carga na camada Bronze')
('silver_produtos', 'data_silver', 'Linhagem', 'Controle', 'Data de carga na camada Silver')
('silver_produtos', 'id_produto', 'Chave Natural', 'Silver', 'Identificador do produto na origem')
('silver_produ

In [5]:
## Via Pandas + SQL
df_dict = pd.read_sql("""
SELECT *
FROM dict_class

""", conexao)

df_dict


,nome_tabela,campo,papel,camada,observacao
0,silver_produtos,id_produto,Chave Natural,Silver,Identificador do produto na origem
1,silver_produtos,nome_produto,Dimensão,Gold,Atributo descritivo do produto
2,silver_produtos,tipo_dado,Dimensão,Gold,Tipo do dado geoespacial
3,silver_produtos,sistema_coordenadas,Dimensão,Gold,Sistema de referência espacial
4,silver_produtos,fornecedor,Dimensão,Gold,Fornecedor do dado
5,silver_produtos,resolucao_espacial,Medida,Gold,Resolução espacial do produto
6,silver_produtos,area_cobertura_km2,Medida,Gold,Área de cobertura em km²
7,silver_produtos,preco,Medida,Gold,Valor comercial do produto
8,silver_produtos,data_bronze,Linhagem,Controle,Data de carga na camada Bronze
9,silver_produtos,data_silver,Linhagem,Controle,Data de carga na camada Silver


In [38]:
# Tabela Dicionário de Metadados: Camada Gold DSC 2 

conexao.execute("""
CREATE TABLE IF NOT EXISTS dict_dim_prod_2 (
    Coluna TEXT,
    Tipo TEXT,
    Descrição TEXT
    
    )""")

conexao.execute("""INSERT INTO dict_dim_prod_2(
    Coluna,
    Tipo,
    Descrição)
    
    VALUES 
    ('sk_produto','INTEGER','Chave substituta (surrogate key)'),
    ('id_produto','INTEGER','Chave natural do produto'),
    ('nome_produto','TEXT','Nome do produto'),
    ('categoria','TEXT','Categoria do produto'),
    ('tipo_dado','TEXT','Tipo de dado Geoespacial'),
    ('sistema_coordenadas','TEXT','Sistema de coordenadas'),
    ('formato','TEXT','Formato do arquivo geoespacial'),
    ('fornecedor','TEXT','Fornecedor do Produto'),
    ('data_inicio','DATE','Data Início Vigência da Versão'),
    ('data_fim','DATE','Data Final Vigência da Versão'),
    ('flag_atual','INTEGER','Indica a versão vigente (1)'),                
    ('data_carga_gold','DATE','Data de carga da camada Gold')                  
""")

conexao.commit()


In [ ]:
# Tabela completa do dicionário de dados Camada Gold SCD 2:

pd.read_sql("""
SELECT *
FROM dict_dim_prod_2;
""", conexao)


,Coluna,Tipo,Descrição
0,sk_produto,INTEGER,Chave substituta (surrogate key)
1,id_produto,INTEGER,Chave natural do produto
2,nome_produto,TEXT,Nome do produto
3,categoria,TEXT,Categoria do produto
4,tipo_dado,TEXT,Tipo de dado Geoespacial
5,sistema_coordenadas,TEXT,Sistema de coordenadas
6,formato,TEXT,Formato do arquivo geoespacial
7,fornecedor,TEXT,Fornecedor do Produto
8,data_inicio,DATE,Data Início Vigência da Versão
9,data_fim,DATE,Data Final Vigência da Versão


In [16]:
# Camada Gold : Tabelas Dimensão Produto
# Estrutura da dimensão Produto ( Camada Gold SDC 2)
# Processo de carga 1: Criação da tabela dimensão 


conexao.execute("""CREATE TABLE IF NOT EXISTS dim_produto_2 (
    sk_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    id_produto INTEGER,

    nome_produto TEXT,
    categoria TEXT,
    tipo_dado TEXT,
    sistema_coordenadas TEXT,
    formato TEXT,
    fornecedor TEXT,

    data_inicio DATE,
    data_fim DATE,
    flag_atual INTEGER,

    data_carga_gold DATE) """ )

# flag_atual = versão vigente                                                                                                                                                         
    
conexao.commit()    
    
    
    
    
                               

In [20]:
# Processo de Carga 2
# Agregar e condicionar versões de dados que sofreram alteração
# Camada GOLD : SCD Tipo 2

conexao.executescript(""" UPDATE dim_produto_2                
    SET 
        data_fim = date('now','-1 day'),
        flag_atual = 0
    WHERE flag_atual = 1
    AND id_produto IN (
        SELECT sp.id_produto
        FROM silver_produtos sp
        JOIN dim_produto dp
            ON sp.id_produto = dp.id_produto
        WHERE 
            flag_atual = 1
            AND (
                sp.nome_produto <> dp.nome_produto OR
                sp.categoria <> dp.categoria OR
                sp.tipo_dado <> dp.tipo_dado OR
                sp.sistema_coordenadas <> dp.sistema_coordenadas OR
                sp.formato <> dp.formato OR
                sp.fornecedor <> dp.fornecedor ));""")   

conexao.commit()                                                
                          

In [27]:
# Camada Gold SCD 2 :
# Processo de carga 3
# Parâmetros para amostragem da nova versão de dados ( produto novo ou alterado)

conexao.executescript("""INSERT INTO dim_produto_2 (
    id_produto,
    nome_produto,
    categoria,
    tipo_dado,
    sistema_coordenadas,
    formato,
    fornecedor,
    data_inicio,
    data_fim,
    flag_atual,
    data_carga_gold
)
SELECT
    sp.id_produto,
    sp.nome_produto,
    sp.categoria,
    sp.tipo_dado,
    sp.sistema_coordenadas,
    sp.formato,
    sp.fornecedor,
    date('now') AS data_inicio,
    NULL AS data_fim,
    1 AS flag_atual,
    date('now') AS data_carga_gold
FROM silver_produtos sp
LEFT JOIN dim_produto dp
    ON sp.id_produto = dp.id_produto
    AND flag_atual = 1
WHERE
    dp.id_produto IS NULL
    OR (
        sp.nome_produto          <> dp.nome_produto OR
        sp.categoria             <> dp.categoria OR
        sp.tipo_dado             <> dp.tipo_dado OR
        sp.sistema_coordenadas   <> dp.sistema_coordenadas OR
        sp.formato               <> dp.formato OR
        sp.fornecedor             <> dp.fornecedor
    );""")

conexao.commit()

# Produto novo → entra
# Produto alterado → nova versão
# Produto igual → ignorado

# A dimensão dim_produto foi implementada utilizando Slowly Changing Dimension Tipo 2 (SCD 2),
# garantindo o versionamento completo de todos os atributos do produto.
# Cada alteração gera um novo registro, preservando o histórico e assegurando rastreabilidade temporal nas análises.


In [28]:
# Tabela completa da dimensão produto:

pd.read_sql("""
SELECT *
FROM dim_produto_2;
""", conexao)


,sk_produto,id_produto,nome_produto,categoria,tipo_dado,sistema_coordenadas,formato,fornecedor,data_inicio,data_fim,flag_atual,data_carga_gold
0,1,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,SIRGAS 2000 / UTM 23S,Shapefile,GeoMapas Ltda,2025-12-24,None,1,2025-12-24
1,2,2,Modelo Digital de Elevação,MDE,Raster,WGS84,GeoTIFF,INPE,2025-12-24,None,1,2025-12-24
2,3,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,SIRGAS 2000,GeoTIFF,Maxar,2025-12-24,None,1,2025-12-24
3,4,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,SIRGAS 2000,GeoPackage,ANA,2025-12-24,None,1,2025-12-24
4,5,5,Classificação de Vegetação Cerrado,Vegetação,Raster,WGS84,GeoTIFF,IBGE,2025-12-24,None,1,2025-12-24
5,6,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,SIRGAS 2000,Shapefile,IBGE,2025-12-24,None,1,2025-12-24
6,7,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,SIRGAS 2000 / UTM 22S,GeoPackage,Defesa Civil,2025-12-24,None,1,2025-12-24


In [30]:
# Mostragem da tabela de dim_produto:

cursor = conexao.cursor()

cursor.execute("SELECT * FROM dim_produto_2;")
rows = cursor.fetchall()

for row in rows:
    print(row)


(1, 1, 'Mapa de Uso do Solo 2023', 'Mapeamento Temático', 'Vetorial', 'SIRGAS 2000 / UTM 23S', 'Shapefile', 'GeoMapas Ltda', '2025-12-24', None, 1, '2025-12-24')
(2, 2, 'Modelo Digital de Elevação', 'MDE', 'Raster', 'WGS84', 'GeoTIFF', 'INPE', '2025-12-24', None, 1, '2025-12-24')
(3, 3, 'Ortoimagem Urbana São Paulo', 'Imagem Orbital', 'Raster', 'SIRGAS 2000', 'GeoTIFF', 'Maxar', '2025-12-24', None, 1, '2025-12-24')
(4, 4, 'Mapa de Drenagem Hidrográfica', 'Hidrografia', 'Vetorial', 'SIRGAS 2000', 'GeoPackage', 'ANA', '2025-12-24', None, 1, '2025-12-24')
(5, 5, 'Classificação de Vegetação Cerrado', 'Vegetação', 'Raster', 'WGS84', 'GeoTIFF', 'IBGE', '2025-12-24', None, 1, '2025-12-24')
(6, 6, 'Limites Administrativos Municipais', 'Base Cartográfica', 'Vetorial', 'SIRGAS 2000', 'Shapefile', 'IBGE', '2025-12-24', None, 1, '2025-12-24')
(7, 7, 'Mapa de Risco de Deslizamento', 'Análise Ambiental', 'Vetorial', 'SIRGAS 2000 / UTM 22S', 'GeoPackage', 'Defesa Civil', '2025-12-24', None, 1, '2025-

In [ ]:
# Overview:
# Quando ocorre alteração em qualquer atributo descritivo do produto:
# A versão vigente é encerrada
# data_fim = data anterior à carga
# flag_atual = 0
# Uma nova versão é inserida
# Novo sk_produto
# Mesma id_produto
# Atributos atualizados
# data_inicio = data atual
# data_fim = NULL
# flag_atual = 1

# Nenhum dado histórico é sobrescrito.